<a href="https://colab.research.google.com/github/sanjay05singh/quiz/blob/main/Quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# Python Quiz Application


import json
import os
import sys
import time
import random

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nGoogle Drive mounted successfully.")

    DRIVE_BASE_PATH = '/content/drive/MyDrive/quiz_folder/'

    if not os.path.exists(DRIVE_BASE_PATH):
        os.makedirs(DRIVE_BASE_PATH)
        print(f"Created directory: {DRIVE_BASE_PATH}")

except ImportError:
    print("Google Colab 'drive' module not found. Running locally or Drive mount failed.")

    DRIVE_BASE_PATH = 'quiz_data/'
    if not os.path.exists(DRIVE_BASE_PATH):
      os.makedirs(DRIVE_BASE_PATH)
    print(f"Quiz data will be saved locally in: {os.path.abspath(DRIVE_BASE_PATH)}")


QUIZ_FILES = {
    "1": ("COMPUTERS", os.path.join(DRIVE_BASE_PATH, "computer.json")),
    "2": ("MATHEMATICS", os.path.join(DRIVE_BASE_PATH, "maths.json")),
    "3": ("SCIENCE", os.path.join(DRIVE_BASE_PATH, "science.json")),
    "4": ("GENERAL", os.path.join(DRIVE_BASE_PATH, "general.json")),
}


def print_separator(char="=", length=40):
  """Prints a separator line."""
  print(char * length)

# --- Quiz Class ---
class Quiz:
    def __init__(self, filename=""):

        self.filename = filename
        self.questions = []
        if filename:
            self._load_quiz()

    def _load_quiz(self):
        """Loads questions from the JSON file (using full path)."""
        filepath = self.filename

        try:
            if os.path.exists(filepath):
                with open(filepath, 'r') as f:
                    # Handle empty file case
                    content = f.read()
                    if not content.strip(): # Check if file is empty or whitespace only
                      self.questions = []
                      print(f"\nInfo: Quiz file '{filepath}' is empty. Starting fresh.")
                    else:
                      self.questions = json.loads(content)
                      self._renumber_questions()
            else:
                # File doesn't exist yet, start empty
                self.questions = []
                print(f"\nInfo: Quiz file '{filepath}' not found. Will be created if questions are added.")
        except (json.JSONDecodeError) as e:
            print(f"\nWarning: Error decoding JSON from file '{filepath}': {e}")
            print("File might be corrupted or not valid JSON. Starting with an empty quiz for this session.")
            self.questions = []
        except (IOError, OSError) as e:
             print(f"\nWarning: Error loading quiz file '{filepath}': {e}")
             print("Starting with an empty quiz for this session.")
             self.questions = []


    def _save_quiz(self):
        """Saves the current questions to the JSON file (using full path)."""
        filepath = self.filename

        try:
            self._renumber_questions()
            # Ensure directory exists before writing
            dir_name = os.path.dirname(filepath)
            if dir_name and not os.path.exists(dir_name):
                print(f"Creating directory: {dir_name}")
                os.makedirs(dir_name)

            with open(filepath, 'w') as f:
                json.dump(self.questions, f, indent=4)
            # print(f"Quiz saved to {filepath}") # Optional confirmation message
        except (IOError, OSError) as e:
            print(f"\nError: Could not save quiz file '{filepath}': {e}")
            print("Check Google Drive permissions or path if applicable.")


    def _renumber_questions(self):
        """Ensures question numbers are sequential (1, 2, 3...)."""
        for i, q in enumerate(self.questions):
            q['ques_no'] = i + 1

    def record_count(self):
        """Returns the number of questions."""
        return len(self.questions)

    def find_record(self, ques_no):
        """Finds a question by its number. Returns the question dict or None."""
        for q in self.questions:
            # Ensure comparison is between integers
            try:
                if q.get('ques_no') == int(ques_no):
                    return q
            except (ValueError, TypeError):
                continue # Skip if ques_no is not valid or conversion fails
        return None

    def display_record(self, ques_no, preview_mode=False):
        """Prints a specific question to the console.
           In preview_mode, finds the record by index if ques_no is temp.
        """
        q = None
        temp_ques_no_display = -1

        if preview_mode and isinstance(ques_no, int) and 1 <= ques_no <= len(self.questions):
             # For previews before saving, use list index temporarily
             # This relies on the calling function adding/removing temp item
             temp_q = self.questions[ques_no - 1]
             # Create a copy to avoid modifying original if needed later
             q = temp_q.copy()
             temp_ques_no_display = ques_no # Display the intended number
        else:
            q = self.find_record(ques_no)
            if q:
                temp_ques_no_display = q.get('ques_no', -1)

        if q:
            print_separator(char="-", length=30)
            print(f"Question # {temp_ques_no_display}")
            print(f"Question : {q.get('ques', 'N/A')}")
            print(f"Answer 1 : {q.get('ans1', 'N/A')}")
            print(f"Answer 2 : {q.get('ans2', 'N/A')}")
            print(f"Answer 3 : {q.get('ans3', 'N/A')}")
            print(f"Solution : {q.get('sol', 'N/A')}")
            print_separator(char="-", length=30)
            return True
        else:
            # Don't print not found during normal preview flow
            if not preview_mode:
              print(f"Question number {ques_no} not found.")
            return False

    def add(self):
        """Adds new questions interactively."""
        if not self.filename:
            print("Error: No quiz file specified.")
            return

        while True:
            print_separator()
            print("--- Add New Question ---")
            next_ques_no = self.record_count() + 1
            print(f"Adding Question # {next_ques_no}")

            # Get Question
            while True:
                ques = input("Enter Question (or 0 to exit add mode): ").strip()
                if ques == '0': return
                if ques: break
                print("Question cannot be empty.")
            ques = ques.upper()

            # Get Answers
            while True:
                ans1 = input("Enter Answer 1: ").strip().upper()
                if ans1: break
                print("Answer cannot be empty.")
            while True:
                ans2 = input("Enter Answer 2: ").strip().upper()
                if ans2: break
                print("Answer cannot be empty.")
            while True:
                ans3 = input("Enter Answer 3: ").strip().upper()
                if ans3: break
                print("Answer cannot be empty.")

            # Get Solution
            while True:
                sol = input("Enter Correct Solution (1, 2, or 3): ").strip()
                if sol in ['1', '2', '3']:
                    break
                print("Invalid input. Please enter 1, 2, or 3.")

            # Confirm Save for this question
            new_question = {
                'ques_no': next_ques_no, # Will be correctly set by _renumber on save
                'ques': ques,
                'ans1': ans1,
                'ans2': ans2,
                'ans3': ans3,
                'sol': sol
            }

            print("\n--- New Question Preview ---")
            # Temporarily add for display_record in preview mode
            self.questions.append(new_question)
            self.display_record(next_ques_no, preview_mode=True)
            self.questions.pop() # Remove the temporary addition

            confirm = input("Save this question (y/n)? ").strip().lower()
            if confirm == 'y':
                self.questions.append(new_question)
                self._save_quiz() # Renumbers and saves
                print("Question added.")
            else:
                print("Question discarded.")

            # Add More?
            more = input("Add another question (y/n)? ").strip().lower()
            if more != 'y':
                break

    def deletion(self):
        """Deletes a question interactively."""
        if not self.filename:
             print("Error: No quiz file specified.")
             return
        if not self.questions:
            print("\nQuiz is empty. Nothing to delete.")
            return

        while True:
            print_separator()
            print("--- Delete Question ---")
            try:
                current_max_q = self.record_count()
                num_str = input(f"Enter question number to delete (1-{current_max_q}) (or 0 to cancel): ").strip()
                if num_str == '0': return
                ques_no_to_delete = int(num_str)
                # Use find_record to check if the *current* number exists
                if self.find_record(ques_no_to_delete):
                    break
                else:
                    print(f"Invalid question number (must be between 1 and {current_max_q}).")
            except ValueError:
                print("Invalid input. Please enter a number.")

        print("\n--- Question to Delete ---")
        if self.display_record(ques_no_to_delete):
            confirm = input("Delete this question (y/n)? ").strip().lower()
            if confirm == 'y':
                # Find the actual index in the list based on the ques_no
                index_to_delete = -1
                for i, q in enumerate(self.questions):
                  if q.get('ques_no') == ques_no_to_delete:
                    index_to_delete = i
                    break

                if index_to_delete != -1:
                  del self.questions[index_to_delete]
                  self._save_quiz() # Renumbers and saves
                  print("Question deleted.")
                else:
                  print("Error: Question not found during deletion process (should not happen).")

                input("Press Enter to continue...")
            else:
                print("Deletion cancelled.")
        else:
             input("Press Enter to continue...") # Pause if not found initially

    def modify(self):
        """Modifies an existing question interactively."""
        if not self.filename:
            print("Error: No quiz file specified.")
            return
        if not self.questions:
            print("\nQuiz is empty. Nothing to modify.")
            return

        while True:
            print_separator()
            print("--- Modify Question ---")
            try:
                current_max_q = self.record_count()
                num_str = input(f"Enter question number to modify (1-{current_max_q}) (or 0 to cancel): ").strip()
                if num_str == '0': return
                ques_no_to_modify = int(num_str)
                # Use find_record to check validity
                if self.find_record(ques_no_to_modify):
                    break
                else:
                     print(f"Invalid question number (must be between 1 and {current_max_q}).")
            except ValueError:
                print("Invalid input. Please enter a number.")

        # Find the actual dictionary reference
        question_to_modify = self.find_record(ques_no_to_modify)

        # Should always be found based on check above, but double-check
        if not question_to_modify:
            print(f"Error: Question {ques_no_to_modify} not found unexpectedly.")
            input("Press Enter to continue...")
            return

        print("\n--- Current Question Data ---")
        self.display_record(ques_no_to_modify)

        confirm_mod = input("Modify this question (y/n)? ").strip().lower()
        if confirm_mod != 'y':
            print("Modification cancelled.")
            return

        # Create a temporary copy to modify, only save if confirmed
        temp_q = question_to_modify.copy()
        modified = False

        # Modify Question Text
        print(f"\nCurrent Question: '{temp_q['ques']}'")
        if input("Modify Question text? (y/n): ").lower() == 'y':
            while True:
                new_ques = input("Enter new Question: ").strip().upper()
                if new_ques:
                    temp_q['ques'] = new_ques
                    modified = True
                    break
                print("Question cannot be empty.")

        # Modify Answers
        for i in range(1, 4):
            ans_key = f'ans{i}'
            print(f"\nCurrent Answer {i}: '{temp_q[ans_key]}'")
            if input(f"Modify Answer {i}? (y/n): ").lower() == 'y':
                while True:
                    new_ans = input(f"Enter new Answer {i}: ").strip().upper()
                    if new_ans:
                        temp_q[ans_key] = new_ans
                        modified = True
                        break
                    print("Answer cannot be empty.")

        # Modify Solution
        print(f"\nCurrent Solution: '{temp_q['sol']}'")
        if input("Modify Solution? (y/n): ").lower() == 'y':
             while True:
                new_sol = input("Enter new Solution (1, 2, or 3): ").strip()
                if new_sol in ['1', '2', '3']:
                    temp_q['sol'] = new_sol
                    modified = True
                    break
                print("Invalid input.")

        if modified:
            print("\n--- Preview Modified Question ---")
            # Need to display the temp version directly
            print_separator(char="-", length=30)
            print(f"Question # {temp_q['ques_no']}") # Show original number
            print(f"Question : {temp_q['ques']}")
            print(f"Answer 1 : {temp_q['ans1']}")
            print(f"Answer 2 : {temp_q['ans2']}")
            print(f"Answer 3 : {temp_q['ans3']}")
            print(f"Solution : {temp_q['sol']}")
            print_separator(char="-", length=30)

            confirm_save = input("Save these changes (y/n)? ").strip().lower()
            if confirm_save == 'y':
                # Find original index and update in self.questions
                for i, q in enumerate(self.questions):
                    if q.get('ques_no') == ques_no_to_modify:
                        self.questions[i] = temp_q # Replace original with modified
                        break
                self._save_quiz() # Renumbers and saves the whole list
                print("Record modified.")
            else:
                print("Changes discarded.")
                # No need to reload, original wasn't changed in self.questions yet
        else:
            print("No changes were made.")

        input("Press Enter to continue...")


    def display_score(self, name, played, correct):
        """Displays the final score."""
        print_separator(length=30)
        print("       S C O R E   B O A R D")
        print_separator(length=30)
        print(f"Player's Name       : {name}")
        print(f"Questions Attempted : {played}")
        print(f"Correct Answers     : {correct}")
        print(f"Wrong Answers       : {played - correct}")
        score = correct * 10
        print(f"Score               : {score}")
        percentage = 0.0
        if played > 0:
            percentage = (correct / played) * 100.0
        print(f"Percentage          : {percentage:.2f}%")
        print_separator(length=30)
        input("\nPress Enter to return to the main menu...")

    def play(self):
        """Starts the quiz playing session."""
        if not self.filename:
            print("Error: No quiz file specified.")
            return
        # Reload questions right before playing to get latest data
        self._load_quiz()
        if not self.questions:
            print(f"\nQuiz '{os.path.basename(self.filename)}' is empty. Cannot play.")
            input("Press Enter to continue...")
            return

        print_separator()
        print("--- Play Quiz ---")
        while True:
            name = input("Enter your name (or 0 to exit): ").strip()
            if name == '0': return
            if name: break
            print("Name cannot be empty.")
        name = name.upper()

        played = 0
        correct = 0
        # Shuffle a copy of the questions for random order
        shuffled_questions = random.sample(self.questions, len(self.questions))

        for q_index, q in enumerate(shuffled_questions):
            print_separator(char="-")
            print(f"SCORE: {correct * 10}  |  Question {q_index + 1} of {len(shuffled_questions)}")
            print(f"(Original Q#: {q.get('ques_no', 'N/A')})") # Show original number for reference
            print(f"Question : {q.get('ques', 'N/A')}")
            print(f"  1 : {q.get('ans1', 'N/A')}")
            print(f"  2 : {q.get('ans2', 'N/A')}")
            print(f"  3 : {q.get('ans3', 'N/A')}")
            print_separator(char="-")

            while True:
                player_ans = input("Enter solution (1/2/3) (or 0 to quit quiz): ").strip()
                if player_ans == '0':
                    print("\nQuiz aborted.")
                    if played > 0:
                        self.display_score(name, played, correct)
                    else:
                        print("No questions were answered.")
                        input("\nPress Enter to return to the main menu...")
                    return
                if player_ans in ['1', '2', '3']:
                    break
                print("Invalid input. Please enter 1, 2, or 3.")

            played += 1
            if player_ans == q.get('sol'):
                correct += 1
                print("\nCORRECT! :)")
                # print("\a") # Beep might not work or be desirable in Colab
            else:
                print(f"\nIncorrect. The correct answer was: {q.get('sol', 'N/A')}")
                # print("\a")
                time.sleep(1.5) # Pause longer to let user see correct answer

            time.sleep(1) # Pause between questions

        print_separator()
        print("--- Quiz Finished ---")
        self.display_score(name, played, correct)


# --- Menu Class ---
class Menu:

    def get_choice(self, prompt, valid_choices):
        """Gets user input and validates against a list of choices."""
        while True:
            choice = input(prompt).strip().lower()
            if choice in valid_choices:
                return choice
            # Provide more specific feedback
            try:
                # Check if it's a number outside the valid range
                num_choice = int(choice)
                valid_nums = [int(vc) for vc in valid_choices if vc.isdigit()]
                if num_choice not in valid_nums:
                     print(f"Invalid number. Please enter one of: {', '.join(valid_choices)}")
                     continue # Skip the generic invalid message if it was an invalid number
            except ValueError:
                 # It wasn't a number or wasn't meant to be
                 pass
            print(f"Invalid choice. Please enter one of: {', '.join(valid_choices)}")


    def start(self):
        """Displays the initial welcome screen."""
        print_separator(length=50)
        print("          C O M P U T E R   Q U I Z")
        print_separator(length=50)
        print("\n       (Python Version for Colab with Google Drive)")
        print("\n" + "="*65)
        print("! NOTE: Quiz data (.json files) are saved to Google Drive.       !")
        print(f"! Ensure the folder '{DRIVE_BASE_PATH}' exists in Drive! !")
        print("="*65 + "\n")
        input("Press Enter to continue...")

    def sub_menu(self):
        """Displays the quiz subject sub-menu and returns the chosen filename."""
        print_separator()
        print("--- Select Quiz Subject ---")
        for key, (name, filename) in QUIZ_FILES.items():
            # Just show the base name for clarity
            print(f"{key}. {name} ({os.path.basename(filename)})")
        print("0. Return to Previous Menu")
        print("-" * 27)

        valid_choices = list(QUIZ_FILES.keys()) + ['0']
        choice = self.get_choice("Enter choice: ", valid_choices)

        if choice == '0':
            return None
        else:
            # Return the full filename path (e.g., "/content/drive/...")
            return QUIZ_FILES[choice][1]

    def edit_menu(self):
        """Displays the edit menu (Delete/Modify)."""
        filename = self.sub_menu()
        if filename is None:
            print("\nReturning to main menu...")
            time.sleep(1)
            return # User chose to return from sub_menu

        quiz = Quiz(filename) # Load the selected quiz

        # Check if file exists / load was successful before showing edit options
        # Allow editing even if file is empty (to potentially add first question via edit?)
        # Let the specific actions (delete/modify) handle empty checks.
        # if not os.path.exists(filename) and not quiz.questions:
        #     print(f"\nQuiz file '{os.path.basename(filename)}' is empty or doesn't exist yet.")
        #     print("Please add questions first using the main menu option.")
        #     input("Press Enter to continue...")
        #     return


        while True:
            print_separator()
            print(f"--- Edit Menu ({os.path.basename(filename)}) ---")
            # Reload count in case changes were made
            current_count = quiz.record_count()
            print(f"Total Questions: {current_count}")
            print("1. Delete Question")
            print("2. Modify Question")
            print("0. Return to Main Menu")
            print("-" * 28)

            choice = self.get_choice("Enter choice: ", ['1', '2', '0'])

            if choice == '1':
                if current_count > 0:
                   quiz.deletion()
                else:
                   print("\nQuiz is empty. Nothing to delete.")
                   time.sleep(1.5)
            elif choice == '2':
                if current_count > 0:
                    quiz.modify()
                else:
                   print("\nQuiz is empty. Nothing to modify.")
                   time.sleep(1.5)
            elif choice == '0':
                break # Exit edit menu loop

    def main_menu(self):
        """Displays the main menu and handles user actions."""
        while True:
            print_separator(length=30)
            print("      M A I N   M E N U")
            print_separator(length=30)
            print("1. Play Quiz")
            print("2. Add Questions")
            print("3. Edit Questions")
            print("0. Quit")
            print("-" * 30)

            choice = self.get_choice("Enter choice: ", ['1', '2', '3', '0'])

            if choice == '1':
                filename = self.sub_menu()
                if filename:
                    quiz = Quiz(filename)
                    quiz.play()
            elif choice == '2':
                filename = self.sub_menu()
                if filename:
                    quiz = Quiz(filename)
                    quiz.add()
            elif choice == '3':
                self.edit_menu()
            elif choice == '0':
                print("\nGoodbye!")
                # In Colab, sys.exit() might raise SystemExit, which is fine
                # Or just break the loop
                break

# --- Main Execution Block ---
if __name__ == "__main__":
    main_menu_instance = Menu()
    main_menu_instance.start()
    main_menu_instance.main_menu()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted successfully.
          C O M P U T E R   Q U I Z

       (Python Version for Colab with Google Drive)

! NOTE: Quiz data (.json files) are saved to Google Drive.       !
! Ensure the folder '/content/drive/MyDrive/quiz_folder/' exists in Drive! !

Press Enter to continue...
      M A I N   M E N U
1. Play Quiz
2. Add Questions
3. Edit Questions
0. Quit
------------------------------
Enter choice: 2
--- Select Quiz Subject ---
1. COMPUTERS (computer.json)
2. MATHEMATICS (maths.json)
3. SCIENCE (science.json)
4. GENERAL (general.json)
0. Return to Previous Menu
---------------------------
Enter choice: 2

Info: Quiz file '/content/drive/MyDrive/quiz_folder/maths.json' not found. Will be created if questions are added.
--- Add New Question ---
Adding Question # 1
Enter Question (or 0 to exit add mode): IF YOU ROLL TWO STANDARD FAIR SIX-SID